In [ ]:
!pip install -U xprof
!pip install -U protobuf
!!pip install tensorboard-plugin-profile

In [1]:
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec as P
Explicit = jax.sharding.AxisType.Explicit
Auto = jax.sharding.AxisType.Auto

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
# Q1
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('X', 'Y'), axis_types=(Auto, Auto))
jax.set_mesh(mesh)
print(jax.devices())

key = jax.random.key(0)
A = jax.random.normal(key, (1024, 1024))
A = jax.device_put(A, P('X', 'Y'))

@jax.jit
@jax.shard_map(in_specs=P('X', 'Y'), out_specs=P('X', 'Y'))
def shmap_avg(A_matrix):
  return jnp.mean(A_matrix, keepdims=True)

@jax.jit
def jit_avg(A_matrix):
  X_axis = mesh.shape['X']
  Y_axis = mesh.shape['Y']
  A_matrix = jnp.permute_dims(
      jnp.reshape(
          A_matrix, 
          (X_axis, A.shape[0] // X_axis, Y_axis, A.shape[1] // Y_axis),
      ), 
      (0, 2, 1, 3)
  )
  A_matrix = jnp.mean(A_matrix, axis=(-2, -1),)
  return A_matrix

jnp.allclose(jit_avg(A), shmap_avg(A))

import functools, numpy as np

def s_roll(x, shift):
  @jax.shard_map(in_specs=P('X', 'Y'), out_specs=P('X', 'Y'))
  def shmap_roll(x,):
    return jnp.roll(x, shift, axis=0)
  return shmap_roll(x)

@functools.partial(jax.jit, static_argnames=['shift'], out_shardings=jax.NamedSharding(mesh, jax.P('X','Y')))
def shift_jit(x, shift: int):
  X, Y = mesh.axis_sizes
  reshaped = x.reshape(X, x.shape[0] // X, -1)
  return jnp.roll(reshaped, shift, axis=1).reshape(x.shape[0], x.shape[1])

x = jnp.arange(8 * 64 * 8, dtype=jnp.float32).reshape(8 * 64, 8)
x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('X','Y')))

y1 = s_roll(x, 5)
y2 = shift_jit(x, 5)

np.testing.assert_array_equal(y1, y2)

E0000 00:00:1783493506.513107    4613 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


In [3]:
# profiling code

jnp.allclose(shmap_avg(A), jit_avg(A))
with jax.profiler.trace("/kaggle/working/"):
    shmap_avg(A).block_until_ready()
    jit_avg(A).block_until_ready()

While profiling, I found that both the jit and the shmap versions of the function seem to lower to the same XLA primitives, such that calling shmap_avg calls the same cached compiled code as the jit code. I verified this by switching the order of the calls in my profiler, and observed that the name of the called function in the profiler changed from jit_avg to shmap_avg. This means that the jit auto paralellism lowers to the same thing as the shmap version.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=/tmp/tensorboard

In [ ]:
# Q2
mesh = jax.make_mesh(axis_shapes=(8,), axis_names='X', axis_types=(Auto))
jax.set_mesh(mesh)

keys = jax.random.split(jax.random.key(2), 6)
E = 8
k = 1
D = 2048
F = 8192
S = 1024
num_devices = 8
BLOCK_SIZE = 512

X = jax.device_put(jax.random.normal(keys[0], (S, D), dtype=jnp.float32), P('X', None))
W = jax.device_put(jax.random.normal(keys[1], (E, D, F), dtype=jnp.float32), P('X', None, None))
routes = jax.device_put(jax.random.categorical(keys[3], jnp.zeros((E)), axis=0, shape=(S, k)), P('X',))

@jax.jit
def moe_basic(X, W, routes):
    out = jnp.zeros((S, F))
    for i in range(E):
        mask = routes[:, 0] == i
        masked = jnp.where(mask[:, None], X, 0).astype(float)
        out += masked @ W[i]
    return out

@jax.jit
def moe_truth(X, W, routes):
  packed_X = jnp.repeat(X, k, axis=0)
  flattened_routes = routes.flatten()
  g = jnp.bincount(flattened_routes, length=E)
  args = jnp.argsort(flattened_routes)
  unsort_args = jnp.argsort(args)

  sorted_X = packed_X[args]
  result = jax.lax.ragged_dot(sorted_X, W, g)

  Y = result[unsort_args]
  Y = Y.reshape(S, k, F)
  Y = Y.sum(axis=1)
  return Y

# @jax.jit
# def moe_local_jit(X, W, routes):
#   # our expert matmul function
#   expert_matmul = jax.vmap(lambda X, W: X @ W, in_axes=(0, 0))

#   # compute some metadata
#   args = jnp.argsort(routes)

#   unsort_args = jnp.argsort(args)
#   sorted_X = X[args]
#   g = jnp.bincount(routes, length=E)
#   g_sum = jnp.cumsum(g)
#   g_offset = (jnp.cumsum(jnp.concatenate((jnp.zeros(1), g[:-1])))).astype(int)
#   blocks = jnp.ceil(jnp.max(g) / BLOCK_SIZE).astype(int)
#   global_padded_expert_size = 32 # jnp.array(BLOCK_SIZE * blocks)

#   # make an [E, pad_size, D] buffer
#   offset_pads = g + ((jnp.arange(E) * blocks) * BLOCK_SIZE)
#   all_indices = jnp.tile(jnp.arange(global_padded_expert_size)[None,:], (E, 1))
#   offset_indices = all_indices + g_offset[:, None]
#   offset_mask = (offset_indices < g_sum[:, None])
#   indices = jnp.where(offset_mask, offset_indices, 0)

#   # [E, Sx, D]
#   X_e = jnp.where(offset_mask[:,:,None], sorted_X[indices], 0)

#   # matmul
#   result = expert_matmul(X_e, W)

#   # unpack and unpad the results
#   unpack_mask = (all_indices < g[:, None])
#   unpack_indices = unpack_mask.flatten().nonzero(size=S)
#   unpacked_results = result.reshape(E * global_padded_expert_size, F)[unpack_indices]
#   Y = unpacked_results[unsort_args]
#   return Y

@jax.jit
@jax.shard_map(mesh=mesh, in_specs=P('X'), out_specs=P('X'))
def moe_shard(X, W, routes):
  # our expert matmul function
  # expert_matmul = jax.vmap(lambda X, W: X @ W, in_axes=(0, 0))

  # compute some metadata
  packed_X = jnp.repeat(X, k, axis=0)
  flattened_routes = routes.flatten()
  args = jnp.argsort(flattened_routes)
  unsort_args = jnp.argsort(args)
  sorted_X = packed_X[args]

  # code from here on shouldnt depend on k at all
  g = jnp.bincount(flattened_routes, length=E)
  g_sum = jnp.cumsum(g)
  g_offset = (jnp.cumsum(jnp.concatenate((jnp.zeros(1), g[:-1])))).astype(int)
  blocks = jnp.ceil(jnp.max(g) / BLOCK_SIZE).astype(int)
  local_padded_expert_size = jnp.array(BLOCK_SIZE * blocks)

  # find the global max across all devices so we can pad evenly
  global_padded_expert_size = jax.lax.pmax(local_padded_expert_size, 'X')

  def cond(carry):
    i, Y_sorted = carry
    return i < global_padded_expert_size

  def body(carry):
    i, Y_sorted = carry

    chunk_idx = i + jnp.arange(BLOCK_SIZE)
    offset_indices = g_offset[:, None] + chunk_idx[None, :]
    offset_mask = chunk_idx[None, :] < g[:, None]
    indices = jnp.where(offset_mask, offset_indices, 0)

    X_e = jnp.where(offset_mask[:, :, None], sorted_X[indices], 0)
    X_commed = jax.lax.all_to_all(X_e, 'X', split_axis=0,concat_axis=0,tiled=False,)

    # local expert matmul
    result = X_commed @ W[0]

    # comms back to the device
    activated = jax.lax.all_to_all(result, 'X', split_axis=0, concat_axis=0,tiled=False,)

    # retrieve and unpack data
    flattened_mask = offset_mask.flatten()[:, None]
    flattened_indices = offset_indices.flatten()[:, None]
    unpacked_mask = jnp.where(flattened_mask, flattened_indices, 0).reshape(E * BLOCK_SIZE)
    unpacked_result = activated.reshape(E * BLOCK_SIZE, F)
    unpacked_result = jnp.where(flattened_mask, unpacked_result, 0)
    Y_sorted = Y_sorted.at[unpacked_mask].add(unpacked_result)
    return i + BLOCK_SIZE, Y_sorted

  # have to think about k again

  # we have to do this because it's a global buffer
  Y_sorted_init = jax.device_put(jnp.zeros((S * k, F)), P('X'))

  # we have to set to varying because we each device accumulates individually
  Y_sorted_init = jax.lax.pcast(Y_sorted_init, 'X', to='varying')
  init_i = jnp.array(0)
  _, Y_sorted = jax.lax.while_loop(cond, body, (init_i, Y_sorted_init))
  Y = Y_sorted[unsort_args]
  Y = Y.reshape(S // num_devices, k, F)
  Y = Y.sum(axis=1)
  return Y

A = jnp.abs(moe_basic(X, W, routes) - moe_shard(X, W, routes))
jnp.max(A)

In [ ]:
# profiling code

moe_basic(X, W, routes)
with jax.profiler.trace("/kaggle/working/"):
    moe_basic(X, W, routes).block_until_ready()

While profiling the basic implementation, I observe that XLA likes to AllGather the entire input matrix into each of the devices. This seems rather useless and inefficient; we should only gather the routed inputs.

In [ ]:
# timing test
%timeit -n 10 -r 100 moe_basic(X, W, routes).block_until_ready()
%timeit -n 10 -r 100 moe_shard(X, W, routes).block_until_ready()

Results of timing experiments
19.7 ms ± 80.7 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (Naive)
7.07 ms ± 99.7 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (Sharded)

We gain a huge speed up, as expected as we save on the naive AllGather from the default auto parallelism.

In [ ]:
x = jnp.arange(64).reshape((8, 8))
@jax.shard_map(mesh=mesh, in_specs=P('X'), out_specs=P('X'))
def all_to_all_testing(x):
  print('Before\n', x, x.shape)
  y = jax.lax.all_to_all(x, 'X', split_axis=1, concat_axis=1, tiled=True)
  y = jax.lax.all_to_all(y, 'X', split_axis=1, concat_axis=1, tiled=True)
  print('After\n', y, y.shape)

all_to_all_testing(x)


In [ ]:
import functools

import jax
jax.config.update('jax_num_cpu_devices', 8)
import jax.numpy as jnp

import numpy as np

In [ ]:
#Q3.1
Explicit = jax.sharding.AxisType.Explicit

# This is intended to run on a TPU v5e-8 runtime. If you can't get this,
# try setting jax.config.update('jax_num_cpu_devices', 8).
#
mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=('X', 'Y'), axis_types=(Explicit, Explicit))
jax.set_mesh(mesh)

B, D, F = 1024, 2048, 8192
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W = jnp.arange(np.prod((D, F))).reshape((D, F))

A = jax.device_put(A, jax.P('X', 'Y'))
W = jax.device_put(W, jax.P('Y', None))

@functools.partial(jax.jit, out_shardings=jax.P('X', None))
def matmul(lhs, rhs):
  return jax.lax.dot(lhs, rhs, out_sharding=jax.P('X', None))

def collective_matmul_all_reduce(lhs, rhs):
  # tile over F ie split F into F_y shards
  axis_size = jax.lax.axis_size('Y')  # axis_size = 4 for this example
  idx = jax.lax.axis_index('Y')

  assert rhs.shape[1] % axis_size == 0
  tile_size = rhs.shape[1] // axis_size

  def f(i, carrys):
    accum, lhs = carrys
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, i * tile_size, tile_size, 1)
    # Matmul for a chunk
    update = lhs @ rhs_chunk
    all_reduced = jax.lax.psum(update, 'Y')
    # AllReduce
    accum = jax.lax.dynamic_update_slice(accum, all_reduced, (0, i * tile_size))
    return accum, lhs

  accum = jnp.zeros((lhs.shape[0], rhs.shape[1]), dtype=lhs.dtype)
  accum = jax.lax.pcast(accum, ('X'), to='varying')
  accum, lhs = jax.lax.fori_loop(0, axis_size, f, (accum, lhs), unroll=True)
  return accum

jit_sharded_f = jax.jit(jax.shard_map(
  collective_matmul_all_reduce,
  in_specs=(jax.P('X', 'Y'), jax.P('Y', None)), out_specs=jax.P('X', None),
  check_vma=True))

shmapped_out = jit_sharded_f(A, W,)
expected_out = matmul(A, W)

np.testing.assert_array_equal(shmapped_out, expected_out)
%timeit -r 100 -n 10 jit_sharded_f(A, W,).block_until_ready()
%timeit -r 100 -n 10 matmul(A, W,).block_until_ready()

1.13 ms ± 22.9 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (sharded)
1.11 ms ± 25.9 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (naive jit)

Because of how XLA handles this, our custom implementation is not a speedup.

In [ ]:
#Q3.2
Explicit = jax.sharding.AxisType.Explicit

mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=('X', 'Y'), axis_types=(Explicit, Explicit))
jax.set_mesh(mesh)

B, D, F = 2048, 4096, 16384
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W = jnp.arange(np.prod((D, F))).reshape((D, F))

A = jax.device_put(A, jax.P('X', 'Y'))
W = jax.device_put(W, jax.P('Y', None))

@functools.partial(jax.jit, out_shardings=jax.P('X', None))
def matmul(lhs, rhs):
  return jax.lax.dot(lhs, rhs, out_sharding=jax.P('X', None))

def collective_matmul_reduce_scatter(lhs, rhs):
  axis_size = jax.lax.axis_size('Y')  # axis_size = 4 for this example
  idx = jax.lax.axis_index('Y')
  chunk_size = rhs.shape[1] // axis_size
  perms = [(j, (j + 1) % axis_size) for j in range(axis_size)]

  def local_matmul(target_idx):
    # this gets a Dy Fy chunk where Fy is given by the target idx
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, target_idx * chunk_size, chunk_size, 1)
    return lhs @ rhs_chunk

  def work(i, carrys):
    target_idx, accum = carrys
    # add the devices contribution to the reduction chunk. at the start this is the source devices contribution to the final
    # reduced chunk of the device that is directly to its left
    accum += local_matmul(target_idx)

    # send the chunk to the **right**
    accum = jax.lax.ppermute(accum, 'Y', perms)

    # the next target is one to the left of the previous target
    target_idx = (target_idx - 1) % axis_size

    return target_idx, accum

  # create buffer for reduction, Bx Fy
  accum = jnp.zeros((lhs.shape[0], rhs.shape[1] // axis_size), dtype=lhs.dtype)
  accum = jax.lax.pcast(accum, ('X', 'Y'), to='varying')

  # the first target device is one to the left of the source device
  target_idx = (idx - 1) % axis_size

  _, accum = jax.lax.fori_loop(0, axis_size - 1, work, (target_idx, accum), unroll = True)

  # add source devices own contribution to its owned chunk
  accum += local_matmul(idx)
  return accum

jit_sharded_cmrs = jax.jit(jax.shard_map(
  collective_matmul_reduce_scatter,
  in_specs=(jax.P('X', 'Y'), jax.P('Y', None)), out_specs=jax.P('X', 'Y'),
  check_vma=True))

shmapped_out = jit_sharded_cmrs(A, W,)
expected_out = matmul(A, W)

np.testing.assert_array_equal(shmapped_out, expected_out)
%timeit -r 100 -n 10 jit_sharded_cmrs(A, W,).block_until_ready()
%timeit -r 100 -n 10 matmul(A, W,).block_until_ready()

3.02 ms ± 84.5 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (sharded implementation)
3.73 ms ± 71.3 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (jit implementation)

Our implementation affords a noticeable speedup compared to auto parallelism. 

In [ ]:
from jax._src.api import block_until_ready
#Q3.3
Explicit = jax.sharding.AxisType.Explicit

mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=('X', 'Y'), axis_types=(Explicit, Explicit))
jax.set_mesh(mesh)

B, D, F = 2048, 4096, 16384
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W1 = jnp.arange(np.prod((D, F))).reshape((D, F))
W2 = jnp.arange(np.prod((D, F))).reshape((F, D))

A = jax.device_put(A, jax.P('X', 'Y'))
W1 = jax.device_put(W1, jax.P(None, 'Y'))
W2 = jax.device_put(W2, jax.P('Y', None))

@functools.partial(jax.jit, out_shardings=jax.P('X', 'Y'))
def transformer_block_ref(A, W1, W2):
  up_proj = A @ W1
  down_proj = jax.lax.dot(up_proj, W2, out_sharding=jax.P('X', 'Y'))
  return down_proj

# We need to pull in the book's AllGather code for the block
def collective_matmul_allgather(lhs, rhs):
  # lhs is the looped operand; rhs is the local operand
  axis_size = jax.lax.axis_size('Y')  # axis_size = 4 for this example
  idx = jax.lax.axis_index('Y')

  chunk_size = lhs.shape[1]
  assert rhs.shape[0] % chunk_size == 0

  def f(i, carrys):
    accum, lhs = carrys
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
    # Matmul for a chunk
    update = lhs @ rhs_chunk
    # Circular shift to the left
    lhs = jax.lax.ppermute(
        lhs,
        axis_name='Y',
        perm=[(j, (j - 1) % axis_size) for j in range(axis_size)]
    )
    return accum + update, lhs

  accum = jnp.zeros((lhs.shape[0], rhs.shape[1]), dtype=lhs.dtype)
  accum = jax.lax.pcast(accum, ('X', 'Y'), to='varying')
  accum, lhs = jax.lax.fori_loop(0, axis_size - 1, f, (accum, lhs), unroll=True)

  # Compute the last chunk after the final permute to leave lhs in the state we found it
  i = axis_size - 1
  rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
  update = lhs @ rhs_chunk
  return accum + update

def collective_transformer_block(A, W1, W2):
  # compute the first projection
  up_proj = collective_matmul_allgather(A, W1)
  down_proj = collective_matmul_reduce_scatter(up_proj, W2)
  return down_proj

jit_sharded_block = jax.jit(jax.shard_map(
  collective_transformer_block,
  in_specs=(jax.P('X', 'Y'), jax.P(None, 'Y'), jax.P('Y', None)), out_specs=jax.P('X', 'Y')))
shmapped_out = jit_sharded_block(A, W1, W2)
expected_out = transformer_block_ref(A, W1, W2)

%timeit -n 10 -r 100 jit_sharded_block(A, W1, W2).block_until_ready()
%timeit -n 10 -r 100 transformer_block_ref(A, W1, W2).block_until_ready()

np.testing.assert_array_equal(shmapped_out, expected_out)

1.13 ms ± 21.1 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (sharded implementation)
1.18 ms ± 28.5 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (naive jit)

The sharded implementaton affords a small but noticeable speedup.

In [ ]:
#Q4 Bidirectional comms

# AllReduce is already bidirectional since psum is bidirectional, so we implement bidirectional ReduceScatter
Explicit = jax.sharding.AxisType.Explicit

mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=('X', 'Y'), axis_types=(Explicit, Explicit))
jax.set_mesh(mesh)

B, D, F = 1024, 2048, 8192
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W = jnp.arange(np.prod((D, F))).reshape((D, F))

A = jax.device_put(A, jax.P('X', 'Y'))
W = jax.device_put(W, jax.P('Y', None))

@functools.partial(jax.jit, out_shardings=jax.P('X', None))
def matmul(lhs, rhs):
  return jax.lax.dot(lhs, rhs, out_sharding=jax.P('X', None))

def bidirectional_collective_matmul_reduce_scatter(lhs, rhs):
  axis_size = jax.lax.axis_size('Y')  # axis_size = 4 for this example
  idx = jax.lax.axis_index('Y')
  chunk_size = rhs.shape[1] // axis_size

  r_perms = [(j, (j + 1) % axis_size) for j in range(axis_size)]
  l_perms = [(j, (j - 1) % axis_size) for j in range(axis_size)]

  def local_matmul(target_idx):
    # this gets a Dy Fy chunk where Fy is given by the target idx
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, target_idx * chunk_size, chunk_size, 1)
    return lhs @ rhs_chunk

  def work(i, carrys):
    left_target_idx, right_target_idx, left_accum, right_accum = carrys

    left_accum += local_matmul(left_target_idx)
    left_accum = jax.lax.ppermute(left_accum, 'Y', l_perms)

    right_accum += local_matmul(right_target_idx)
    right_accum = jax.lax.ppermute(right_accum, 'Y', r_perms)

    right_target_idx = (right_target_idx - 1) % axis_size
    left_target_idx = (left_target_idx + 1) % axis_size

    return left_target_idx, right_target_idx, left_accum, right_accum

  # create buffer for reduction, Bx Fy
  left_accum = jnp.zeros((lhs.shape[0], rhs.shape[1] // axis_size), dtype=lhs.dtype)
  left_accum = jax.lax.pcast(left_accum, ('X', 'Y'), to='varying')
  right_accum = jnp.zeros((lhs.shape[0], rhs.shape[1] // axis_size), dtype=lhs.dtype)
  right_accum = jax.lax.pcast(right_accum, ('X', 'Y'), to='varying')

  # the first target devices are the furthest from the source
  right_target_idx = (idx + (axis_size // 2)) % axis_size
  left_target_idx = (idx + (axis_size // 2)) % axis_size

  # do the first iteration outside of loop bc in the first iteration we only send to the right
  right_accum += local_matmul(right_target_idx)
  right_accum = jax.lax.ppermute(right_accum, 'Y', r_perms)
  right_target_idx = (right_target_idx - 1) % axis_size
  left_target_idx = (left_target_idx + 1) % axis_size

  _, _, left_accum, right_accum = jax.lax.fori_loop(0, (axis_size // 2) - 1, work, (left_target_idx, right_target_idx, left_accum, right_accum), unroll = True)

  # add source devices own contribution to its owned chunk
  accum = local_matmul(idx) + left_accum + right_accum
  return accum

jit_sharded_b = jax.jit(jax.shard_map(
  bidirectional_collective_matmul_reduce_scatter,
  in_specs=(jax.P('X', 'Y'), jax.P('Y', None)), out_specs=jax.P('X', 'Y'),
  check_vma=True))

shmapped_out = jit_sharded_b(A, W,)

np.testing.assert_array_equal(shmapped_out, expected_out)
%timeit -r 100 -n 10 jit_sharded_b(A, W,).block_until_ready()
%timeit -r 100 -n 10 jit_sharded_cmrs(A, W,).block_until_ready()

1 ms ± 26.1 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (bidirectional)
1.02 ms ± 23.9 μs per loop (mean ± std. dev. of 100 runs, 10 loops each) (unidirectional)

This is expected, since the Y axis size is so small that the difference between unidirectional and bidirectional is not substantial.

In [ ]:
!pip install -U xprof
!pip install -U protobuf
%load_ext tensorboard
%tensorboard --logdir=/tmp/tensorboard

In [ ]:
jnp.array(0)